# Deteção de Anomalias em Transações em Python

> Relatório do projeto desafio da DIO sobre Detecção de Anomalias em Transações em Python

## 1. Introdução

Modelos preditivos são técnicas utilizadas para identificar padrões em dados históricos e, a partir desses padrões, realizar previsões ou classificações sobre novos dados. Eles são aplicados em diferentes áreas para apoiar a tomada de decisão, a avaliação de riscos e a identificação de comportamentos que merecem atenção. A modelagem preditiva pode ser utilizada, por exemplo, em avaliação de crédito, previsão de vendas e detecção de fraudes (QuestionPro).

No contexto deste projeto, o problema é tratado como uma tarefa de classificação binária, na qual cada transação deve ser classificada como normal (`0`) ou fraudulenta (`1`). Esse tipo de aplicação é particularmente relevante porque transações fraudulentas representam uma parcela muito pequena do conjunto de dados. Em situações de forte desbalanceamento, uma métrica como acurácia, isoladamente, pode transmitir uma impressão equivocada sobre o desempenho do modelo. A própria documentação da W3Schools demonstra que uma acurácia elevada pode ocorrer mesmo quando o modelo apresenta desempenho inadequado para uma das classes, tornando métricas como precision, recall, F1-score e AUC importantes para a avaliação (W3Schools – AUC/ROC).

A detecção de anomalias também possui relação direta com esse problema. Anomalias são observações que se afastam do comportamento considerado normal e sua identificação pode ser aplicada à detecção de transações fraudulentas, inclusive em conjuntos de dados com classes altamente desbalanceadas (Built In).

### Modelos utilizados

Foram avaliados cinco modelos no notebook: **Regressão Logística, Random Forest, XGBoost, Isolation Forest e SVM (Support Vector Machine)**.

* **Regressão Logística:** é um algoritmo de classificação utilizado para estimar a probabilidade de uma observação pertencer a uma determinada categoria. No caso binário, o modelo produz uma probabilidade associada à classe positiva e, a partir de um limiar, transforma essa probabilidade em uma classificação (W3Schools – Logistic Regression). No projeto, foi utilizada tanto a versão com todas as variáveis quanto uma versão com conjunto reduzido de características.

* **Random Forest:** é um método de conjunto (ensemble) que combina diversas árvores de decisão para produzir uma previsão mais robusta e reduzir o risco de sobreajuste. A abordagem é apresentada como uma técnica de classificação e regressão em materiais sobre modelagem preditiva (QuestionPro). No notebook, foi utilizado um modelo com 50 árvores, profundidade máxima de 10 e class_weight="balanced".

* **XGBoost:** é um método baseado em gradient boosting, no qual diferentes árvores são construídas de forma sequencial para melhorar o desempenho do conjunto. Essa família de métodos é destacada entre as técnicas de modelagem preditiva (QuestionPro; Keyrus). No projeto, foi utilizado scale_pos_weight=10 para auxiliar no tratamento do desbalanceamento.

* **Isolation Forest:** é um algoritmo de detecção de anomalias não supervisionado que procura isolar observações atípicas por meio de divisões sucessivas. Como as anomalias tendem a ser isoladas mais rapidamente que observações normais, o método produz um escore de anomalia que pode ser utilizado para identificar possíveis fraudes (Built In). No notebook, a proporção de contaminação foi definida de acordo com a proporção observada da classe de fraude.

* **SVM:** as Máquinas de Vetores de Suporte são métodos supervisionados utilizados principalmente em classificação e também podem ser aplicados a regressão e detecção de outliers. O SVC utilizado no projeto pertence ao conjunto de implementações de SVM disponibilizadas pelo scikit-learn (scikit-learn). Foi utilizado kernel linear, C=1, class_weight="balanced" e estimativa de probabilidades.



## 2. Objetivo

O objetivo do projeto é desenvolver e comparar modelos de aprendizado de máquina capazes de identificar transações potencialmente fraudulentas, avaliando seu desempenho por meio de métricas adequadas a um problema de classificação desbalanceado.

Além de comparar os modelos, buscou-se compreender como a escolha do algoritmo e do limiar de classificação interfere nos resultados e identificar uma alternativa mais adequada ao contexto de detecção de fraudes financeiras.

## 3. Metodologia

### 3.1 Ambiente e apoio utilizado

O desenvolvimento do código foi realizado no Google Colab, ambiente utilizado para execução e experimentação do notebook em Python. Durante a elaboração do projeto, houve também auxílio da IA Gemini, principalmente como ferramenta de apoio para compreensão, construção e ajustes do código.

O código completo utilizado no desenvolvimento está disponibilizado no arquivo `deteccao_de_anomalias_em_transacoes_em_python.ipynb` que acompanha este relatório no repositório. Por esse motivo, esta seção apresenta a metodologia e as principais etapas realizadas, sem reproduzir o código.

### 3.2 Conjunto de dados

Foi utilizado o conjunto de dados de transações de cartão de crédito disponibilizado no arquivo `creditcard.csv`, carregado no notebook a partir do endereço:

`https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv`

O conjunto possui **284.807 transações** e **31 colunas** originais. A variável Class representa o resultado da transação:

*   `0`: transação normal;
*   `1`: transação fraudulenta.

A distribuição encontrada no notebook demonstra forte desbalanceamento:

*   **99,8273%** das transações são normais;
*   **0,1727%** são fraudulentas.

Não foram identificados dados ausentes nas colunas do conjunto analisado.

As variáveis `V1` a `V28` são características numéricas disponibilizadas no conjunto de dados, enquanto `Time`, `Amount` e `Class` representam, respectivamente, o tempo, o valor da transação e a variável-alvo.

### 3.3 Preparação dos dados

Durante o pré-processamento, foram realizadas as seguintes etapas:

1. carregamento do conjunto de dados com pandas;

2. verificação da estrutura e da existência de valores ausentes;

3. análise da proporção entre as classes;

4. criação da variável Amount_log, utilizando uma transformação logarítmica do valor da transação;

5. criação da variável Amount_scaled, obtida pela padronização de Amount com StandardScaler;

6. separação entre variáveis preditoras (X) e variável-alvo (y);

7. divisão dos dados em treinamento e teste utilizando train_test_split, com 80% para treinamento e 20% para teste, mantendo a proporção das classes por meio de stratify=y.

A divisão entre treinamento e teste é importante para avaliar o comportamento do modelo em dados que não foram utilizados durante seu treinamento. Esse procedimento é uma prática básica de avaliação de modelos de aprendizado de máquina (W3Schools – Train/Test). Ademais, em problemas extremamente desbalanceados (como `0,17%` de fraudes), sem o parâmetro `stratify`, corre-se o risco da amostra de teste não conter transações fraudulentas suficientes para garantir validade estatística na avaliação.

O conjunto completo utilizado nos principais modelos possui 32 características. Também foi criado um conjunto reduzido contendo apenas `V1` a `V28` e `Amount_scaled`, totalizando 29 características. A versão reduzida foi utilizada posteriormente para avaliar novamente a Regressão Logística e o SVM.

### 3.4 Treinamento e avaliação

Foram treinados e avaliados os cinco modelos citados anteriormente. Para cada modelo foram analisados:

* matriz de confusão;

* precision;

* recall;

* F1-score;

* acurácia;

* curva ROC;

* AUC-ROC;

* curva Precision-Recall.

A matriz de confusão permite observar os quatro tipos de resultado de uma classificação: verdadeiro positivo, verdadeiro negativo, falso positivo e falso negativo (W3Schools – Confusion Matrix).

Após o treinamento inicial dos modelos supervisionados, também foi realizada uma **otimização do limiar de classificação**, testando 100 valores entre 0 e 1 e escolhendo aquele que maximizou o F1-score. Essa etapa foi aplicada à Regressão Logística, Random Forest, XGBoost e SVM.



## 4. Resultados

### 4.1 Comparação geral

Os principais resultados obtidos no conjunto de teste estão apresentados abaixo. Os valores correspondem às execuções registradas no notebook antes das otimizações.

| Modelo | Precision – fraude | Recall – fraude | F1-score – fraude | Acurácia | AUC-ROC |
| :---: | :---: | :---: | :---: | :---: | :---: |
| Regressão Logística | 0,82 | 0,72 | 0,77 | 1,00 | 0,9585 |
| Random Forest | 0,82 | 0,82 | 0,82 | 1,00 | 0,9585 |
| XGBoost | 0,89 | 0,84 | 0,86 | 1,00 | 0,9734 |
| Isolation Forest | 0,23 | 0,24 | 0,24 | 1,00 | 0,9518 |
| SVM | 0,00 | 0,78 | 0,00 | 0,45 | 0,3689 |


A primeira observação importante é que a acurácia de 1,00 não deve ser utilizada como principal critério de escolha. Como 99,8273% das transações são normais, um modelo pode obter uma acurácia muito alta mesmo apresentando dificuldade para identificar fraudes. Por isso, neste problema, recall, F1-score, matriz de confusão e AUC são mais informativos. Essa interpretação é consistente com a explicação apresentada pela W3Schools sobre o uso de métricas alternativas à acurácia em problemas nos quais as classes não estão equilibradas (W3Schools – AUC/ROC).

### 4.2 Regressão Logística

Na execução inicial, a Regressão Logística apresentou:

* precision para fraude: **0,82**;

* recall: **0,72**;

* F1-score: **0,77**;

* AUC: **0,9585**.

O recall de 0,72 significa que o modelo identificou aproximadamente 72% das fraudes presentes no conjunto de teste. Consequentemente, aproximadamente 28% das fraudes não foram detectadas.

Com o limiar otimizado em **0,4444**, o recall passou para **0,73** e o F1-score permaneceu em **0,77**. A alteração demonstra que o limiar pode influenciar a relação entre falsos positivos e falsos negativos.

Além da Regressão Logística treinada com o conjunto completo de características, o notebook também avaliou uma segunda versão utilizando um conjunto reduzido, composto pelas variáveis `V1` a `V28` e `Amount_scaled`. O objetivo dessa etapa foi verificar o comportamento do modelo utilizando uma quantidade menor de características, além de utilizar a `Amount_scaled`, visto que o modelo é sensível a diferentes escalas numéricas.

Com o limiar padrão de classificação, a Regressão Logística com características reduzidas apresentou:

* precision para fraude: **0,83**;

* recall: **0,64**;

* F1-score: **0,72**;

* AUC-ROC: **0,95597**;

* acurácia: **1,00**.

Em comparação com a Regressão Logística utilizando todas as características, houve uma redução do recall, de 0,72 para 0,64, indicando que a utilização do conjunto reduzido fez com que uma parcela maior das fraudes deixasse de ser identificada quando utilizado o limiar padrão.

Posteriormente, foi realizada novamente a otimização do limiar de classificação, buscando maximizar o F1-score. O limiar considerado ótimo foi **0,1515**. Com essa alteração, os resultados para a classe fraude passaram para:

* precision: **0,74**;

* recall: **0,77**;

* F1-score: **0,75**;

* AUC-ROC: **0,95597**;

* acurácia: **1,00**.

A alteração do limiar aumentou o recall de **0,64** para **0,77**, reduzindo, portanto, a quantidade de falsos negativos. Por outro lado, a precision caiu de **0,83** para **0,74**, mostrando o trade-off entre identificar uma parcela maior das fraudes e aumentar a quantidade de transações normais classificadas como suspeitas.

Esse resultado é especialmente relevante para o problema estudado, pois demonstra que a definição do limiar pode ser tão importante quanto a escolha do modelo. Como o custo de um falso negativo é elevado no contexto de fraudes financeiras, aumentar o recall pode ser uma estratégia interessante, desde que o aumento dos falsos positivos permaneça operacionalmente aceitável.

A Regressão Logística apresenta a vantagem de ser relativamente simples de interpretar e adequada a problemas de classificação binária (W3Schools – Logistic Regression). Entretanto, apresentou desempenho inferior ao Random Forest e ao XGBoost neste conjunto de dados.

### 4.3 Random Forest

O Random Forest apresentou:

* precision para fraude: **0,82**;

* recall: **0,82**;

* F1-score: **0,82**;

* AUC: **0,9585**.

Com a otimização do limiar para **0,4242**, o recall aumentou para **0,84**, o F1-score para **0,83** e AUC para **0,9698**.

Esse resultado é superior ao da Regressão Logística em termos de identificação de fraudes. O modelo também apresentou bom equilíbrio entre precision e recall, característica relevante para o cenário analisado.

### 4.4 XGBoost

O XGBoost apresentou o melhor desempenho geral entre os modelos avaliados. Os resultados foram:

* precision para fraude: **0,89**;

* recall: **0,84**;

* F1-score: **0,86**;

* AUC: **0,9734**;

* limiar otimizado: **0,5051**.

O modelo conseguiu manter um recall elevado e, ao mesmo tempo, apresentar a maior precision entre os modelos com desempenho adequado. Isso significa que, além de detectar uma parcela maior das fraudes, o XGBoost também apresentou menor quantidade relativa de falsos positivos entre as transações classificadas como fraude.

O AUC de aproximadamente 0,9734 também foi o maior observado no experimento. A métrica AUC permite avaliar a capacidade do modelo de separar as classes em diferentes limiares, sendo especialmente útil quando a acurácia não representa adequadamente o desempenho (W3Schools – AUC/ROC).

### 4.5 Isolation Forest

O Isolation Forest apresentou:

* precision para fraude: **0,23**;

* recall: **0,24**;

* F1-score: **0,24**;

* AUC: **0,9518**.

Apesar de apresentar uma AUC relativamente alta, o desempenho na classificação efetiva das fraudes foi consideravelmente inferior ao dos modelos supervisionados. O recall de 0,24 indica que apenas cerca de 24% das fraudes foram identificadas.

Esse resultado é coerente com a natureza do algoritmo. O Isolation Forest procura identificar observações que se comportam como anomalias sem utilizar diretamente os rótulos de fraude durante o treinamento. Essa característica é uma vantagem quando não existem dados rotulados, mas, neste experimento, os modelos supervisionados conseguiram explorar melhor as informações disponíveis (Built In).

### 4.6 SVM

O SVM apresentou o resultado mais problemático entre os modelos avaliados. Na versão com todas as características, foram observados:

* precision para fraude: **0,00**;

* recall: **0,78**;

* F1-score: **0,00**;

* acurácia: **0,45**;

* AUC: **0,3689**.


Foi realizado o treinamento na versão com características reduzidas, visto que o modelo é sensível a variáveis com diferentes magnitudes; o SVM também apresentou desempenho inadequado, com AUC de 0,4664. A otimização do limiar chegou a 0,0000, produzindo uma classificação que praticamente não é útil para o objetivo do projeto. Em ambos treinamentos houveram problemas de convergência dos dados.

O resultado reforça que a escolha do algoritmo precisa considerar não apenas sua capacidade teórica, mas também o comportamento observado nos dados, o tratamento das classes e a configuração do modelo. O scikit-learn disponibiliza diferentes implementações de SVM e destaca sua aplicação em problemas de classificação e detecção de outliers (scikit-learn – Support Vector Machines).

## 5. Análise dos falsos positivos e falsos negativos

No contexto de transações financeiras, os tipos de erro não possuem o mesmo custo.

Um **falso positivo (FP)** ocorre quando uma transação normal é classificada como fraude. Embora esse erro possa causar inconvenientes ao cliente, existem mecanismos para realizar uma validação adicional, como envio de e-mail, token, autenticação adicional ou CAPTCHA.

Já um **falso negativo (FN)** ocorre quando uma transação fraudulenta é classificada como normal. Nesse caso, a transação pode ser autorizada e o prejuízo decorrente da fraude pode ser muito mais difícil de recuperar.

Por esse motivo, o objetivo não deve ser simplesmente maximizar a acurácia. É necessário priorizar um modelo capaz de apresentar a menor quantidade possível de falsos negativos, mantendo, sempre que possível, um nível aceitável de falsos positivos.

A relação entre recall e falsos negativos é direta: o recall é calculado como TP / (TP + FN), portanto, quanto maior o recall da classe fraude, menor tende a ser a quantidade de fraudes não detectadas (W3Schools – Confusion Matrix).

Considerando os 98 casos de fraude presentes no conjunto de teste:

* XGBoost: recall de 0,84, correspondendo a aproximadamente 16 fraudes não detectadas;

* Random Forest: recall de 0,84 após otimização, também correspondendo a aproximadamente 16 falsos negativos;

* SVM: recall de 0,78, aproximadamente 22 falsos negativos, mas com precision praticamente nula;

* Regressão Logística: recall de 0,73 após otimização, aproximadamente 26 falsos negativos;

* Isolation Forest: recall de 0,24, aproximadamente 74 falsos negativos.

Assim, XGBoost e Random Forest apresentaram a menor quantidade estimada de falsos negativos entre os modelos que tiveram desempenho consistente. Entre os dois, o XGBoost apresentou vantagem por alcançar a mesma faixa de recall, mas com maior precision, maior F1-score e maior AUC.

É importante destacar que a otimização realizada no notebook buscou maximizar o F1-score, e não diretamente minimizar o número de falsos negativos. Portanto, em uma aplicação financeira real, seria recomendável definir o limiar a partir de uma função de custo de negócio que atribua um peso explicitamente maior aos falsos negativos.

## 6. Conclusão

A análise realizada demonstrou que o problema de detecção de fraudes exige uma avaliação diferente daquela utilizada em problemas de classificação equilibrados. A forte predominância de transações normais faz com que a acurácia, apesar de elevada em vários modelos, não seja suficiente para determinar qual algoritmo é mais adequado.

Entre os modelos avaliados, o **XGBoost** apresentou o melhor resultado geral. O modelo alcançou AUC de **0,9734**, precision de **0,89**, recall de **0,84** e F1-score de **0,86** para a classe fraude. Além disso, apresentou aproximadamente 16 falsos negativos entre as 98 fraudes presentes no conjunto de teste, mesma faixa alcançada pelo Random Forest após a otimização do limiar.

O Random Forest também apresentou desempenho competitivo e pode ser considerado uma alternativa relevante, principalmente por ter alcançado recall de **0,84** após a otimização. Entretanto, o XGBoost apresentou melhor precision, F1-score e AUC, tornando-se a escolha mais adequada para este experimento.

A Regressão Logística apresentou desempenho razoável e serve como uma boa referência de modelo de classificação, mas ficou abaixo dos modelos de conjunto. O Isolation Forest demonstrou que uma abordagem de detecção de anomalias não supervisionada pode ser útil, porém não apresentou desempenho suficiente para este conjunto específico. O SVM, por sua vez, apresentou resultados insatisfatórios, não sendo recomendado com a configuração utilizada.

Por fim, o projeto permitiu compreender, na prática, a importância do pré-processamento, da divisão entre treinamento e teste, do tratamento do desbalanceamento de classes, da escolha de métricas adequadas e da definição do limiar de classificação. Também evidenciou que a escolha de um modelo de detecção de fraude deve considerar o custo de cada tipo de erro, e não apenas uma métrica agregada de desempenho.

Em um cenário real de instituição financeira, uma próxima etapa seria aprofundar a otimização do limiar do XGBoost com base em uma função de custo de negócio, atribuindo penalidade maior aos falsos negativos. Também seria importante avaliar o modelo com validação cruzada, monitoramento de drift dos dados e testes em dados mais recentes antes de uma eventual utilização em produção.

## 7. Referências

* QuestionPro. Modelos preditivos: dos dados à tomada de decisão inteligente. Disponível em: https://www.questionpro.com/blog/pt-br/modelos-preditivos/

* W3Schools. Machine Learning – Train/Test. Disponível em: https://www.w3schools.com/python/python_ml_train_test.asp

* W3Schools. Machine Learning – Confusion Matrix. Disponível em: https://www.w3schools.com/python/python_ml_confusion_matrix.asp

* W3Schools. Machine Learning – Logistic Regression. Disponível em: https://www.w3schools.com/python/python_ml_logistic_regression.asp

* W3Schools. Python Machine Learning – AUC – ROC Curve. Disponível em: https://www.w3schools.com/python/python_ml_auc_roc.asp

* Built In. 8 Anomaly Detection Algorithms to Know. Disponível em: https://builtin.com/machine-learning/anomaly-detection-algorithms

* Keyrus. As 11 técnicas mais utilizadas na modelagem de analítica preditiva. Disponível em: https://keyrus.com/pt/pt/insights/as-11-tecnicas-mais-utilizadas-na-modelagem-de-analitica-preditiva

* scikit-learn. Support Vector Machines. Disponível em: https://scikit-learn.org/stable/modules/svm.html

* Materiais Didáticos DIO.